In [1]:
import json
import time
import numpy as np
import pandas as pd

from uniguide.config import get_path, load_config
from uniguide.evaluation.protocol import evaluate_model
from uniguide.models.content_based import ContentBasedRecommender
from uniguide.models.embeddings import ContentFusionRecommender, EmbeddingContentRecommender

cfg = load_config()
p = get_path("processed")
tables = get_path("results") / "tables"
train, val, test = (pd.read_parquet(p / f"{n}.parquet") for n in ["train", "val", "test"])
courses = pd.read_parquet(p / "courses.parquet")
cold_courses = set(json.loads((p / "cold_courses.json").read_text()))
courses_warm = courses[~courses["course_id"].isin(cold_courses)].reset_index(drop=True)
K = tuple(cfg["evaluation"]["k_values"])
names = courses.set_index("course_id")[["name", "institution"]]


def mean_metric(res, metric, k):
    return res[(res["metric"] == metric) & (res["k"] == k)]["value"].mean()

In [2]:
t = time.time()
emb = EmbeddingContentRecommender().fit(train, courses_warm)
print(f"{time.time() - t:.0f}s", emb.item_vectors.shape)   # (n_courses, 384)

cid = "machine-learning"
sim = emb.item_vectors @ emb.item_vectors[emb.item_index[cid]]
best = np.argsort(-sim)[1:6]
pd.DataFrame({"course_id": [emb.catalog[i] for i in best], "cosine": sim[best]}).join(names, on="course_id")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

76s (380, 384)


,course_id,cosine,name,institution
0,machine-learning-duke,0.854948,Introduction to Machine Learning,Duke University
1,uol-machine-learning-for-all,0.841027,Machine Learning for All,University of London
2,ml-foundations,0.827226,Machine Learning Foundations: A Case Study App...,University of Washington
3,python-machine-learning,0.815987,Applied Machine Learning in Python,University of Michigan
4,machine-learning-with-python,0.793667,Machine Learning with Python,IBM


In [3]:
sample_users = val["user_id"].drop_duplicates().sample(3000, random_state=cfg["seed"])
val_s = val[val["user_id"].isin(sample_users)]

configs = {
    "tfidf (reference)": ContentBasedRecommender(),
    "emb name_weight=0.0": EmbeddingContentRecommender(name_weight=0.0),
    "emb name_weight=0.5": EmbeddingContentRecommender(name_weight=0.5),
    "emb name_weight=1.0": EmbeddingContentRecommender(name_weight=1.0),
    "fusion tfidf+emb 50/50": ContentFusionRecommender(
        [ContentBasedRecommender(), EmbeddingContentRecommender(name_weight=0.5)]),
}
rows = []
for label, model in configs.items():
    model.fit(train, courses_warm)
    res = evaluate_model(model, val_s, k_values=K)
    rows.append({"model": label, "ndcg@10": mean_metric(res, "ndcg", 10),
                 "hit_rate@10": mean_metric(res, "hit_rate", 10)})
tuning_emb = pd.DataFrame(rows)
tuning_emb.to_csv(tables / "tuning_embeddings.csv", index=False)
tuning_emb

,model,ndcg@10,hit_rate@10
0,tfidf (reference),0.105430,0.192321
1,emb name_weight=0.0,0.085204,0.169570
2,emb name_weight=0.5,0.104706,0.211518
3,emb name_weight=1.0,0.107495,0.201209
4,fusion tfidf+emb 50/50,0.116526,0.218272


In [4]:
NAME_WEIGHT = 0.5   # <- best value from the table above
train_val = pd.concat([train, val])

final_models = {
    "eval_content_embeddings": EmbeddingContentRecommender(name_weight=NAME_WEIGHT),
    "eval_content_fusion": ContentFusionRecommender(
        [ContentBasedRecommender(), EmbeddingContentRecommender(name_weight=NAME_WEIGHT)]),
}
for file_name, model in final_models.items():
    model.fit(train_val, courses_warm)
    res = evaluate_model(model, test, k_values=K, experiment="main", seed=cfg["seed"])
    res.to_csv(tables / f"{file_name}.csv", index=False)
    print(model.name, res.attrs)
    display(res.groupby(["k", "metric"])["value"].mean().unstack())

content_embeddings {'evaluated_users': 14039, 'excluded_users_without_relevant_items': 717}


metric,hit_rate,ndcg,precision,recall
k,,,,
5,0.145666,0.07743,0.030586,0.110236
10,0.218392,0.09704,0.023563,0.167420


content_fusion {'evaluated_users': 14039, 'excluded_users_without_relevant_items': 717}


metric,hit_rate,ndcg,precision,recall
k,,,,
5,0.162262,0.090480,0.033906,0.126848
10,0.225800,0.107673,0.023990,0.177646
